# Drinking Water Potability

**Tabular Classification Project** — Predict whether water is safe to drink.

Models: CatBoost · LightGBM · XGBoost  
Baselines: LazyPredict · FLAML AutoML  
Dataset: Water Potability (3,276 rows × 10 columns, local CSV)  
Target: `Potability` (0 = not potable, 1 = potable)

## 2 · Project Overview

We predict whether drinking water meets quality standards based on 9 chemical/physical measurements. This is a **binary classification** task with significant class imbalance and missing values — a realistic data-quality challenge common in environmental monitoring.

## 3 · Learning Objectives

1. Handle missing values in water-quality measurements.
2. Address class imbalance in environmental classification.
3. Compare gradient-boosting models on messy real-world data.
4. Evaluate with appropriate metrics (F1, ROC-AUC, precision/recall).
5. Use LazyPredict and FLAML for rapid model comparison.

## 4 · Problem Statement

Given 9 water-quality measurements (pH, hardness, solids, chloramines, sulfate, conductivity, organic carbon, trihalomethanes, turbidity), predict whether the water is **potable** (safe to drink).

## 5 · Why This Project Matters

- Clean drinking water is a global health priority.
- Automated water-quality classification can supplement manual testing.
- This dataset teaches handling of real-world data quality issues (missing values, imbalance).

## 6 · Dataset Overview

| Property | Value |
|----------|-------|
| **Rows** | 3,276 |
| **Columns** | 10 (9 features + 1 target) |
| **Features** | ph, Hardness, Solids, Chloramines, Sulfate, Conductivity, Organic_carbon, Trihalomethanes, Turbidity |
| **Target** | Potability (0/1) |
| **Missing** | ph, Sulfate, Trihalomethanes have significant missing values |

## 7 · Dataset Source and License Notes

- **Source**: Kaggle (water potability dataset).
- **Local file**: `water_potability.csv`.
- **License**: Public / educational.
- **Limitations**: Simplified measurements — real water testing involves many more parameters.

## 8 · Environment Setup

In [ ]:
import subprocess, sys

def _install(pkg, imp=None):
    try: __import__(imp or pkg)
    except ImportError: subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

for p, i in [("catboost",None),("lightgbm",None),("xgboost",None),("flaml",None),("lazypredict",None)]:
    _install(p, i)
print("All packages ready.")

## 9 · Imports

In [ ]:
import warnings, os, re, json
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix, classification_report,
                             roc_auc_score, ConfusionMatrixDisplay)
from sklearn.linear_model import LogisticRegression

SAVE_DIR = os.path.dirname(os.path.abspath("__file__"))
SEED = 42
np.random.seed(SEED)
print("Core imports loaded.")

## 10 · Configuration / Constants

In [ ]:
TARGET = "Potability"
TEST_SIZE = 0.2
LOCAL_CSV = os.path.join(SAVE_DIR, "water_potability.csv")
print(f"Target: {TARGET}, CSV: {LOCAL_CSV}")

## 11 · Dataset Download or Loading

In [ ]:
if os.path.exists(LOCAL_CSV):
    df = pd.read_csv(LOCAL_CSV)
    print(f"Loaded from local: {LOCAL_CSV}")
else:
    import kagglehub
    path = kagglehub.dataset_download("adityakadiwal/water-potability")
    import glob
    csv_files = glob.glob(os.path.join(path, "**", "*.csv"), recursive=True)
    df = pd.read_csv(csv_files[0])
    print("Loaded from kagglehub")

print(f"Shape: {df.shape}")
print(df.head())

## 12 · Data Validation Checks

In [ ]:
print(f"Missing values:\n{df.isnull().sum()}")
print(f"\nMissing %:\n{(df.isnull().sum() / len(df) * 100).round(1)}")
print(f"\nDuplicates: {df.duplicated().sum()}")
print(f"\nData types:\n{df.dtypes}")
print(f"\nBasic stats:\n{df.describe()}")

## 13 · Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(16, 12))
num_cols = [c for c in df.columns if c != TARGET]
for i, c in enumerate(num_cols):
    ax = axes[i // 3, i % 3]
    df[c].hist(bins=30, ax=ax, color="steelblue", edgecolor="white")
    ax.set_title(c)
plt.suptitle("Feature Distributions", fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "eda_distributions.png"), dpi=100)
plt.show()

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(df.corr(), annot=True, fmt=".2f", cmap="coolwarm", ax=ax)
ax.set_title("Correlation Matrix")
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "eda_correlation.png"), dpi=100)
plt.show()

## 14 · Target Analysis

In [ ]:
print(f"Target distribution:\n{df[TARGET].value_counts()}")
print(f"\nProportions:\n{df[TARGET].value_counts(normalize=True)}")
fig, ax = plt.subplots(figsize=(6, 4))
df[TARGET].value_counts().plot.bar(ax=ax, color=["steelblue", "coral"])
ax.set_title("Potability Distribution")
ax.set_xticklabels(["Not Potable (0)", "Potable (1)"], rotation=0)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "eda_target.png"), dpi=100)
plt.show()

## 15 · Train/Validation/Test Split Strategy

In [ ]:
X = df.drop(columns=[TARGET])
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=SEED, stratify=y
)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")

## 16 · Preprocessing Strategy

We median-impute missing values (fit on train only) and sanitize column names.

In [ ]:
from sklearn.impute import SimpleImputer
imputer = SimpleImputer(strategy="median")
X_train = pd.DataFrame(imputer.fit_transform(X_train), columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(imputer.transform(X_test), columns=X_test.columns, index=X_test.index)

X_train.columns = [re.sub(r"[^A-Za-z0-9_]", "_", str(c)) for c in X_train.columns]
X_test.columns = [re.sub(r"[^A-Za-z0-9_]", "_", str(c)) for c in X_test.columns]

print(f"NaN remaining: Train={X_train.isnull().sum().sum()}, Test={X_test.isnull().sum().sum()}")

## 17 · Feature Engineering

In [ ]:
for df_part in [X_train, X_test]:
    df_part["ph_Hardness"] = df_part["ph"] * df_part["Hardness"]
    df_part["Solids_Conductivity"] = df_part["Solids"] / (df_part["Conductivity"] + 1)
    df_part["Chlor_Organic"] = df_part["Chloramines"] * df_part["Organic_carbon"]
print(f"Features: {X_train.shape[1]}")

## 18 · Baseline Model

In [ ]:
baseline = LogisticRegression(max_iter=1000, random_state=SEED, class_weight="balanced")
baseline.fit(X_train, y_train)
bl_pred = baseline.predict(X_test)
print("Baseline — Logistic Regression:")
print(f"  Accuracy: {accuracy_score(y_test, bl_pred):.4f}")
print(f"  F1:       {f1_score(y_test, bl_pred, average='weighted'):.4f}")

## 19 · LazyPredict Benchmark

In [ ]:
from lazypredict.Supervised import LazyClassifier

lp = LazyClassifier(verbose=0, ignore_warnings=True, random_state=SEED)
lp_models, lp_preds = lp.fit(X_train, X_test, y_train, y_test)
print(lp_models.head(20))

## 20 · FLAML AutoML Run

In [ ]:
try:
    from flaml import AutoML
    automl = AutoML()
    automl.fit(X_train, y_train, task="classification", time_budget=60,
               metric="accuracy", verbose=0, seed=SEED)
    flaml_pred = automl.predict(X_test)
    print(f"FLAML best: {automl.best_estimator}")
    print(f"  Accuracy : {accuracy_score(y_test, flaml_pred):.4f}")
    print(f"  F1       : {f1_score(y_test, flaml_pred, average='weighted'):.4f}")
except Exception as e:
    print(f"FLAML skipped: {e}")

## 21 · Additional Best-Library Workflow

## 22 · Top 3 Model Selection

## 23 · Final Training and Evaluation of Top 3

In [ ]:
import torch
from catboost import CatBoostClassifier
import lightgbm as lgb
import xgboost as xgb

device = "gpu" if torch.cuda.is_available() else "cpu"
xgb_device = "cuda" if torch.cuda.is_available() else "cpu"

n_classes = len(np.unique(y_train))
if n_classes == 2:
    n_pos = int(y_train.sum()); n_neg = int(len(y_train) - n_pos)
    spw = n_neg / max(n_pos, 1)
else:
    spw = 1.0

models = {
    "CatBoost": CatBoostClassifier(iterations=500, depth=6, learning_rate=0.05,
                                    auto_class_weights="Balanced",
                                    task_type="GPU" if device=="gpu" else "CPU",
                                    verbose=0, random_seed=SEED),
    "LightGBM": lgb.LGBMClassifier(n_estimators=500, learning_rate=0.05,
                                    is_unbalance=True, device=device,
                                    verbose=-1, random_state=SEED),
    "XGBoost": xgb.XGBClassifier(n_estimators=500, learning_rate=0.05,
                                  scale_pos_weight=spw if n_classes==2 else 1.0,
                                  device=xgb_device,
                                  eval_metric="logloss", verbosity=0, random_state=SEED),
}

results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    acc = accuracy_score(y_test, preds)
    f1 = f1_score(y_test, preds, average="weighted")
    results[name] = {"accuracy": acc, "f1": f1}
    print(f"{name}: Acc={acc:.4f} F1={f1:.4f}")

res_df = pd.DataFrame(results).T.sort_values("f1", ascending=False)
print("\nRanking:")
print(res_df)

## 24 · Error Analysis

In [ ]:
best_name = res_df.index[0]
best_model = models[best_name]
preds_best = best_model.predict(X_test)

cm = confusion_matrix(y_test, preds_best)
disp = ConfusionMatrixDisplay(confusion_matrix=cm)
fig, ax = plt.subplots(figsize=(6, 5))
disp.plot(ax=ax, cmap="Blues")
ax.set_title(f"Confusion Matrix — {best_name}")
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "confusion_matrix.png"), dpi=100)
plt.show()

print(classification_report(y_test, preds_best))

errors = X_test.copy()
errors["true"] = y_test.values
errors["pred"] = preds_best
errors["wrong"] = errors["true"] != errors["pred"]
print(f"Total errors: {errors['wrong'].sum()} / {len(errors)} ({errors['wrong'].mean()*100:.1f}%)")
print("\nSample misclassifications:")
print(errors[errors["wrong"]].head(10))

## 25 · Interpretation and Business Insight

- Water potability is hard to predict from these features alone — no single feature is strongly discriminative.
- Feature interactions (pH × Hardness) help marginally.
- In practice, additional measurements (E. coli, lead, arsenic) would greatly improve predictions.

## 26 · Limitations

- Limited feature set.
- Significant missing values.
- Class imbalance affects model performance.

## 27 · How to Improve

1. Collect more features.
2. Try cross-validation.
3. Use SMOTE for class imbalance.
4. Add domain-specific thresholds.

## 28 · Production Considerations

- Water safety classification should never rely solely on ML.
- Regulatory compliance requires specific threshold-based testing.
- Model should supplement, not replace, laboratory testing.

## 29 · Common Mistakes

1. Ignoring missing values.
2. Not handling class imbalance.
3. Using accuracy alone on imbalanced data.
4. Pre-split imputation (data leakage).

## 30 · Mini Challenge

1. Try SMOTE before training.
2. Optimize the classification threshold for recall.
3. Run 5-fold CV and compare stability.

## 31 · Final Summary

In [ ]:
metrics = {}
for name, m in results.items():
    metrics[name] = m
best = res_df.index[0]
metrics["best_model"] = best
metrics["best_accuracy"] = results[best]["accuracy"]
metrics["best_f1"] = results[best]["f1"]

with open(os.path.join(SAVE_DIR, "metrics.json"), "w") as f:
    json.dump(metrics, f, indent=2)
print("Metrics saved to metrics.json")
print(json.dumps(metrics, indent=2))
print("\n✅ Drinking Water Potability notebook complete.")